# TrOCR-small (EN + RU): сборка + расширение словаря

Точная small-архитектура HF; грузятся все английские веса, словарь расширяется русскими
токенами (английские строки эмбеддингов сохраняются).

Нужно: `pip install transformers` (скачает small чекпойнт). Двуязычный токенайзер:
`python scripts/train_tokenizer.py --ru-text-dirs <папки с рус .txt>` (если нет — берётся английский).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from transformers import AutoTokenizer

TOK_DIR = ROOT / 'assets' / 'tokenizer_bi'
if TOK_DIR.exists():
    tokenizer = AutoTokenizer.from_pretrained(str(TOK_DIR)); print('двуязычный токенайзер:', TOK_DIR)
else:
    print('нет tokenizer_bi -> английский (расшир. словарь: scripts/train_tokenizer.py)')
    tokenizer = AutoTokenizer.from_pretrained('microsoft/trocr-small-handwritten')
print('vocab size:', len(tokenizer))

## Сборка модели (грузит EN веса, расширяет словарь под RU)

In [ ]:
from src.model import build_trocr_small, build_processor

model, report = build_trocr_small(tokenizer)
print(report.summary())
print('параметров:', round(sum(p.numel() for p in model.parameters()) / 1e6, 1), 'M')
processor = build_processor(tokenizer)
model.eval();

## Пример 1: шаг обучения (forward + loss) на синтетике

In [ ]:
from IPython.display import display
from src.synth import HandwrittenLineGenerator, make_generator

gen = HandwrittenLineGenerator.from_dirs(
    ru_text_dirs=[], en_text_dirs=[],
    ru_font_dirs=str(ROOT / 'assets' / 'fonts_ru'),
    en_font_dirs=str(ROOT / 'assets' / 'fonts_en'), p_ru=0.5, curriculum=False)
img, text = gen.sample(make_generator(0, 0, 0))
print('текст:', repr(text)); display(img)

pixel_values = processor(images=img, return_tensors='pt').pixel_values
labels = processor.tokenizer(text, return_tensors='pt').input_ids
out = model(pixel_values=pixel_values, labels=labels)
print('pixel_values:', tuple(pixel_values.shape), '| loss:', round(float(out.loss), 3))
out.loss.backward(); print('backward ok')

## Пример 2: инференс (необученная под RU -> мусор, важен пайплайн)

In [ ]:
import torch
with torch.no_grad():
    ids = model.generate(pixel_values, max_new_tokens=32)
print('предсказание:', processor.tokenizer.batch_decode(ids, skip_special_tokens=True))

Сохранить заготовку: `model.save_pretrained('outputs/init'); processor.save_pretrained('outputs/init')`.
Дальше — `scripts/run_pretrain.py` (синтетика EN+RU) -> finetune на реальных строках.